In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("../data/HousePricePrediction.csv")
df.head()

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0


In [3]:
df.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           4
LotArea            0
LotConfig          0
BldgType           0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
Exterior1st        1
BsmtFinSF2         1
TotalBsmtSF        1
SalePrice       1459
dtype: int64

In [4]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'LotConfig', 'BldgType',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'Exterior1st', 'BsmtFinSF2',
       'TotalBsmtSF', 'SalePrice'],
      dtype='object')

In [5]:
df["MSZoning"].nunique()

5

In [6]:
df["MSZoning"]=df["MSZoning"].fillna(df["MSZoning"].mode()[0])


In [7]:

df.head()

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0


In [8]:
df["Exterior1st"].nunique()


15

In [9]:
df["Exterior1st"] = df["Exterior1st"].fillna(df["Exterior1st"].mode()[0])

In [10]:
df["BsmtFinSF2"] = df["BsmtFinSF2"].fillna(0)
df["TotalBsmtSF"] = df["TotalBsmtSF"].fillna(df["TotalBsmtSF"].median())

In [11]:
df = df.dropna(subset=["SalePrice"])

In [12]:
df.isnull().sum()

Id              0
MSSubClass      0
MSZoning        0
LotArea         0
LotConfig       0
BldgType        0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
Exterior1st     0
BsmtFinSF2      0
TotalBsmtSF     0
SalePrice       0
dtype: int64

In [13]:
current_year = 2025

df["HouseAge"] = current_year - df["YearBuilt"]
df["RemodelAge"] = current_year - df["YearRemodAdd"]
df["TotalSpace"] = df["LotArea"] + df["TotalBsmtSF"]
df["HasSecondBasement"] = (df["BsmtFinSF2"] > 0).astype(int)


In [14]:
df = pd.get_dummies(df, columns=["MSZoning","LotConfig","BldgType","Exterior1st"])

In [15]:
df=df.drop(columns="Id")

In [16]:
X=df.drop(columns=["SalePrice"])
y=df["SalePrice"]

In [17]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [19]:
from sklearn.linear_model import RidgeCV, LassoCV

In [20]:
from sklearn.linear_model import RidgeCV, LassoCV
alphas = [0.001,0.01,0.1,1,10,50,100]
ridge_model = RidgeCV(
    alphas=alphas,
    cv=5
)

ridge_model.fit(X_train, y_train)

print("Best alpha for Ridge:", ridge_model.alpha_)

Best alpha for Ridge: 10.0


In [23]:
ridge_pred = ridge_model.predict(X_test)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
print("Ridge R2:", r2_score(y_test, ridge_pred))
print("Ridge MAE:", mean_absolute_error(y_test, ridge_pred))
print("Ridge RMSE:", np.sqrt(mean_squared_error(y_test, ridge_pred)))

Ridge R2: 0.6215517452428081
Ridge MAE: 33972.605783606596
Ridge RMSE: 53877.83928971464


In [24]:
lasso_model = LassoCV(
    alphas=alphas,
    cv=5,
    max_iter=10000,
    random_state=42
)

lasso_model.fit(X_train, y_train)

print("Best alpha for Lasso:", lasso_model.alpha_)

Best alpha for Lasso: 100.0


In [25]:
lasso_pred = lasso_model.predict(X_test)

print("Lasso R2:", r2_score(y_test, lasso_pred))
print("Lasso MAE:", mean_absolute_error(y_test, lasso_pred))
print("Lasso RMSE:", np.sqrt(mean_squared_error(y_test, lasso_pred)))

Lasso R2: 0.6217087124407747
Lasso MAE: 34031.47439295846
Lasso RMSE: 53866.66480223262


In [26]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

In [27]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
print("R2 Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print(mean_absolute_percentage_error(y_test, y_pred))

R2 Score: 0.6259666514999098
MAE: 33935.87555504811
RMSE: 53562.6529903271
0.19622847107309901
